In [8]:
import numpy as np
import pandas as pd 
import math

In [9]:
a = np.array([[1, 2], [3, 4]])
np.std(a)

1.118033988749895

In [10]:
with open(r'C:\Users\Stefanos Rigas\Documents\Projects\Thesis\thesis\robustness-evaluation\explanations\G_E_MASHAP\features.txt','r') as file:
    data = file.readlines()

data[6] = 'Run number: 2\n'
data[14] = 'Run number: 3\n'
data[22] = 'Run number: 4\n'
data[30] = 'Run number: 5\n'
data[38] = ''

# Fix the names of the runs
with open(r'C:\Users\Stefanos Rigas\Documents\Projects\Thesis\thesis\robustness-evaluation\explanations\G_E_MASHAP\features.txt','w') as file:
    file.writelines( data )
    
file.close()

# Extract the explanations from the different runs

with open(r'C:\Users\Stefanos Rigas\Documents\Projects\Thesis\thesis\robustness-evaluation\explanations\G_E_MASHAP\features.txt','r') as file:
    data = file.readlines()

file.close()
    
# Make script that compares the  data - explanations, and the highlighted features (if top feature is the same)

def get_change(current, previous):
    
    if current == previous:
        return 100.0
    try:
        change = (abs(current - previous) / ((current + previous)/2) ) * 100.0
        return "{:.2f}".format(change)
    except ZeroDivisionError:
        return 0

values = []
features = []
for i in range(5):
    for j in range(5):
        x = data[i*8+j].split()
        values.append(x[2])
        features.append(x[0]+x[1])
        
values = np.array(values)
sub_arrays = np.hsplit(values, 5)
df = pd.DataFrame(sub_arrays)
df.index = ['Run_1','Run_2','Run_3','Run_4','Run_5']
df.columns = ['1st Feature', '2nd Feature', '3rd Feature', '4th Feature', '5th Feature']

features = np.array(features)
sub_arrays2 = np.hsplit(features, 5)
df2 = pd.DataFrame(sub_arrays2)
df2.index = ['Run_1','Run_2','Run_3','Run_4','Run_5']
df2.columns = ['1st Feature', '2nd Feature', '3rd Feature', '4th Feature', '5th Feature']

if df2.loc["Run_1","1st Feature"] == df2.loc["Run_2","1st Feature"]:
    print(get_change(float(df.loc["Run_1","1st Feature"]),float(df.loc["Run_2","1st Feature"])),'%')
else:
    print(df2)
    
# Make a df where I compare 1st feature 4 times (1st vs 2nd run, 2nd vs 3rd etc.)

       1st Feature  2nd Feature  3rd Feature  4th Feature  5th Feature
Run_1  "Feature23"  "Feature20"  "Feature13"  "Feature27"   "Feature2"
Run_2  "Feature20"  "Feature23"  "Feature13"  "Feature10"  "Feature18"
Run_3  "Feature23"  "Feature13"  "Feature28"   "Feature1"   "Feature7"
Run_4  "Feature23"  "Feature20"  "Feature10"  "Feature13"  "Feature18"
Run_5  "Feature23"  "Feature13"  "Feature10"  "Feature20"  "Feature21"


In [11]:
df2 = df2.applymap(lambda x: x.replace('"', ''))

In [12]:
uniqueValues = (df2['1st Feature'].append(df2['2nd Feature'].append(df2['3rd Feature'].append(df2['4th Feature'].append(df2['5th Feature']))))).unique()
uniqueValues

array(['Feature23', 'Feature20', 'Feature13', 'Feature28', 'Feature10',
       'Feature27', 'Feature1', 'Feature2', 'Feature18', 'Feature7',
       'Feature21'], dtype=object)

In [13]:
len(uniqueValues)

11

In [14]:
df2 = df2.astype(str)

In [15]:
df2.describe()

,1st Feature,2nd Feature,3rd Feature,4th Feature,5th Feature
count,5,5,5,5,5
unique,2,3,3,5,4
top,Feature23,Feature20,Feature13,Feature27,Feature18
freq,4,2,2,1,2


In [16]:
df2[df2 == 'Feature23']

,1st Feature,2nd Feature,3rd Feature,4th Feature,5th Feature
Run_1,Feature23,NaN,NaN,NaN,NaN
Run_2,NaN,Feature23,NaN,NaN,NaN
Run_3,Feature23,NaN,NaN,NaN,NaN
Run_4,Feature23,NaN,NaN,NaN,NaN
Run_5,Feature23,NaN,NaN,NaN,NaN


In [17]:
df1 = df.astype(float)

In [18]:
dict = {animal: df1[df2.eq(animal)].stack().values for animal in np.unique(df2)}

In [19]:
dict

{'Feature1': array([0.5171281]),
 'Feature10': array([0.76619977, 0.9761319 , 0.7082221 ]),
 'Feature13': array([0.8940995 , 0.76794565, 0.89276755, 0.68825036, 0.73072785]),
 'Feature18': array([0.38098046, 0.6061422 ]),
 'Feature2': array([0.57249504]),
 'Feature20': array([1.1082137, 2.3441222, 1.1134199, 0.6463849]),
 'Feature21': array([0.62242085]),
 'Feature23': array([2.822946 , 2.308442 , 3.7594023, 3.0115108, 3.7976658]),
 'Feature27': array([0.78857476]),
 'Feature28': array([0.6091183]),
 'Feature7': array([0.4641101])}

### Calculate std deviation for each feature

In [20]:
matrix = []
matrix2 = []
matrix3= []

In [21]:
for i in dict:
    matrix.append(i)
    matrix2.append(dict[i])

In [22]:
for list in matrix2:
    matrix3.append(np.std(list, dtype=np.float64))

In [23]:
matrix3

[0.0,
 0.11508864716720947,
 0.08442422059243604,
 0.11258086999999997,
 0.0,
 0.6302700089873046,
 0.0,
 0.5700359522543202,
 0.0,
 0.0,
 0.0]

In [24]:
# Calculate the relative standard deviation (percentage)
matrix4 = []
for list in matrix2:
    matrix4.append(str("{:.2f}".format(100*np.std(list, dtype=np.float64)/len(list)))+"%")

In [25]:
matrix4

['0.00%',
 '3.84%',
 '1.69%',
 '5.63%',
 '0.00%',
 '15.76%',
 '0.00%',
 '11.40%',
 '0.00%',
 '0.00%',
 '0.00%']

In [26]:
def getList(dict):
    feature_names = []
    for key in dict.keys():
        feature_names.append(key)
          
    return feature_names
keys = getList(dict)

In [27]:
del list

In [28]:
z = list(zip(keys, matrix4))

In [29]:
z

[('Feature1', '0.00%'),
 ('Feature10', '3.84%'),
 ('Feature13', '1.69%'),
 ('Feature18', '5.63%'),
 ('Feature2', '0.00%'),
 ('Feature20', '15.76%'),
 ('Feature21', '0.00%'),
 ('Feature23', '11.40%'),
 ('Feature27', '0.00%'),
 ('Feature28', '0.00%'),
 ('Feature7', '0.00%')]

### Measure ranking deviation

In [30]:
df2.columns

Index(['1st Feature', '2nd Feature', '3rd Feature', '4th Feature',
       '5th Feature'],
      dtype='object')

In [31]:
def getIndexes(dfObj, value):
     
    # Empty list
    listOfPos = []
     
    # isin() method will return a dataframe with
    # boolean values, True at the positions   
    # where element exists
    result = dfObj.isin([value])
     
    # any() method will return
    # a boolean series
    seriesObj = result.any()
 
    # Get list of column names where
    # element exists
    columnNames = list(seriesObj[seriesObj == True].index)
    
    # Iterate over the list of columns and
    # extract the row index where element exists
    for col in columnNames:
        rows = list(result[col][result[col] == True].index)
 
        for row in rows:
            listOfPos.append((row, col))
             
    # This list contains a list tuples with
    # the index of element in the dataframe
    return listOfPos
 
# Calling getIndexes() function to get
# the index positions of all occurrences
# of 22 in the dataframe
listOfPositions = getIndexes(df, 22)

In [32]:
listOfPositions = []
for item in uniqueValues:
    listOfPositions.append(str(item) + str(getIndexes(df2, item)))   

In [33]:
listOfPositions

["Feature23[('Run_1', '1st Feature'), ('Run_3', '1st Feature'), ('Run_4', '1st Feature'), ('Run_5', '1st Feature'), ('Run_2', '2nd Feature')]",
 "Feature20[('Run_2', '1st Feature'), ('Run_1', '2nd Feature'), ('Run_4', '2nd Feature'), ('Run_5', '4th Feature')]",
 "Feature13[('Run_3', '2nd Feature'), ('Run_5', '2nd Feature'), ('Run_1', '3rd Feature'), ('Run_2', '3rd Feature'), ('Run_4', '4th Feature')]",
 "Feature28[('Run_3', '3rd Feature')]",
 "Feature10[('Run_4', '3rd Feature'), ('Run_5', '3rd Feature'), ('Run_2', '4th Feature')]",
 "Feature27[('Run_1', '4th Feature')]",
 "Feature1[('Run_3', '4th Feature')]",
 "Feature2[('Run_1', '5th Feature')]",
 "Feature18[('Run_2', '5th Feature'), ('Run_4', '5th Feature')]",
 "Feature7[('Run_3', '5th Feature')]",
 "Feature21[('Run_5', '5th Feature')]"]

In [34]:
# In how many columns the feature occurs
len(listOfPositions[8].split(', ')) / 2

2.0

In [35]:
listOfPositions2 = {}
for item in uniqueValues:
    listOfPositions2[item] = getIndexes(df2, item)

In [36]:
for item in uniqueValues:
    print(listOfPositions2["%s"%item])

[('Run_1', '1st Feature'), ('Run_3', '1st Feature'), ('Run_4', '1st Feature'), ('Run_5', '1st Feature'), ('Run_2', '2nd Feature')]
[('Run_2', '1st Feature'), ('Run_1', '2nd Feature'), ('Run_4', '2nd Feature'), ('Run_5', '4th Feature')]
[('Run_3', '2nd Feature'), ('Run_5', '2nd Feature'), ('Run_1', '3rd Feature'), ('Run_2', '3rd Feature'), ('Run_4', '4th Feature')]
[('Run_3', '3rd Feature')]
[('Run_4', '3rd Feature'), ('Run_5', '3rd Feature'), ('Run_2', '4th Feature')]
[('Run_1', '4th Feature')]
[('Run_3', '4th Feature')]
[('Run_1', '5th Feature')]
[('Run_2', '5th Feature'), ('Run_4', '5th Feature')]
[('Run_3', '5th Feature')]
[('Run_5', '5th Feature')]


In [37]:
c = df2.columns[(df2 == 'Feature23').any()].tolist()
d = df2.value_counts()

In [38]:
matrix = []
for col in df2.columns:
    matrix.append(df2[col].value_counts())

In [39]:
matrix

[Feature23    4
 Feature20    1
 Name: 1st Feature, dtype: int64,
 Feature20    2
 Feature13    2
 Feature23    1
 Name: 2nd Feature, dtype: int64,
 Feature13    2
 Feature10    2
 Feature28    1
 Name: 3rd Feature, dtype: int64,
 Feature27    1
 Feature10    1
 Feature1     1
 Feature13    1
 Feature20    1
 Name: 4th Feature, dtype: int64,
 Feature18    2
 Feature2     1
 Feature7     1
 Feature21    1
 Name: 5th Feature, dtype: int64]

In [40]:
magic = []
keys = []
rankings = {}
for item in uniqueValues:
    new = []
    for col in df2.columns:
        for row in df2.index:
            if df2.at[row,col] == item:
                new.append(int(str(col).split(' ')[0][0]))
    magic.append(new)

In [41]:
magic

[[1, 1, 1, 1, 2],
 [1, 2, 2, 4],
 [2, 2, 3, 3, 4],
 [3],
 [3, 3, 4],
 [4],
 [4],
 [5],
 [5, 5],
 [5],
 [5]]

In [42]:
rankings = []
for list in magic:
    rankings.append(np.std(list, dtype=np.float64))

In [43]:
ranking_deviation = []
for i in range(len(uniqueValues)):
    ranking_deviation.append(uniqueValues[i]+": "+str(rankings[i]))

In [44]:
ranking_deviation

['Feature23: 0.4',
 'Feature20: 1.0897247358851685',
 'Feature13: 0.7483314773547882',
 'Feature28: 0.0',
 'Feature10: 0.4714045207910317',
 'Feature27: 0.0',
 'Feature1: 0.0',
 'Feature2: 0.0',
 'Feature18: 0.0',
 'Feature7: 0.0',
 'Feature21: 0.0']

In [45]:
rankings_percentage = []
for list in magic:
    rankings_percentage.append(str("{:.2f}".format(100*np.std(list, dtype=np.float64)/len(list)))+"%")

In [46]:
ranking_deviation_percentage = []
for i in range(len(uniqueValues)):
    ranking_deviation_percentage.append(uniqueValues[i]+": "+str(rankings_percentage[i]))

In [47]:
ranking_deviation_percentage

['Feature23: 8.00%',
 'Feature20: 27.24%',
 'Feature13: 14.97%',
 'Feature28: 0.00%',
 'Feature10: 15.71%',
 'Feature27: 0.00%',
 'Feature1: 0.00%',
 'Feature2: 0.00%',
 'Feature18: 0.00%',
 'Feature7: 0.00%',
 'Feature21: 0.00%']

In [48]:
df2

,1st Feature,2nd Feature,3rd Feature,4th Feature,5th Feature
Run_1,Feature23,Feature20,Feature13,Feature27,Feature2
Run_2,Feature20,Feature23,Feature13,Feature10,Feature18
Run_3,Feature23,Feature13,Feature28,Feature1,Feature7
Run_4,Feature23,Feature20,Feature10,Feature13,Feature18
Run_5,Feature23,Feature13,Feature10,Feature20,Feature21


TEST: MORE RUNS

In [49]:
with open(r'C:\Users\Stefanos Rigas\Documents\Projects\Thesis\thesis\robustness-evaluation\explanations\G_E_MASHAP\features.txt','r') as file:
    data = file.readlines()

In [50]:
experiments = 10

In [51]:
from math import ceil
j=0
for i in range(6,8*experiments,8):
    j+=1
    data[i] = 'Run number: '+str(j)+'\n'

In [52]:
# Fix the names of the runs
with open(r'C:\Users\Stefanos Rigas\Documents\Projects\Thesis\thesis\robustness-evaluation\explanations\G_E_MASHAP\features.txt','w') as file:
    file.writelines( data )
    
file.close()

TEST: CHANGE INPUTS

In [53]:
import sys
import pandas as pd
import numpy as np
import shap
from xgboost.sklearn import XGBClassifier
from sklearn.metrics import r2_score
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

In [54]:
X, y = load_breast_cancer(return_X_y=True)

In [55]:
from sklearn.neural_network import MLPClassifier
# train original model and get its predictions
X, y = load_breast_cancer(return_X_y=True)
X_train, _, y_train, _ = train_test_split(X, y, random_state=42)
original_model = MLPClassifier()
original_model.fit(X_train, y_train)
predictions = original_model.predict(X)

In [56]:
from sklearn.neural_network import MLPClassifier
# train original model and get its predictions
X, y = load_breast_cancer(return_X_y=True)
X_train, _, y_train, _ = train_test_split(X, y, random_state=42)

Think of when to implement the input_change script and how to measure the differences - important experiment

In [57]:
# script for making small changes to the model input

def input_earthquake(numpy_array, earthquake_magnitude):
    import random
    for i in range(len(numpy_array)):
        n = random.uniform(0.005*earthquake_magnitude, 0.01*earthquake_magnitude)	
        if random.choice([0,1]):
            temp = numpy_array[i] * n
            numpy_array[i] = numpy_array[i] - temp
        else:
            temp = numpy_array[i] * n
            numpy_array[i] = numpy_array[i] + temp

def change_inputs(training_data, earthquake_magnitude):
    for i in range(len(training_data)):
        input_earthquake(training_data[i],earthquake_magnitude)
    return "Earthquake is over"

In [58]:
Z = np.array([[2.12, 8.12, 9.12],[0.12, 2.24, 4.12],[5.12, 6.12, 7.12]])
Z_old= Z.copy()
Z.shape

(3, 3)

In [59]:
change_inputs(Z,10) # input training data and earthquake magnitude (1: 0.5-1% change, 10: 5-10% change)

'Earthquake is over'

In [60]:
# check how much our array elements changed
import numpy as np
subtraction = np.subtract(Z_old, Z)
addition = np.add(Z_old, Z)
result = (abs(subtraction) / (addition/2))*100
print ('Change of each element after the earthquake (percentage):', result)

Change of each element after the earthquake (percentage): [[ 8.89426232  5.32094027  6.26263528]
 [10.38751481 10.16488274  6.52865344]
 [ 6.9625335   6.12672508  5.43796494]]


In [61]:
for i in range(2):
    print(i)

0
1


In [62]:
# check if there are elements that are the same | 1.0 = No
error = np.mean( Z_old != Z )
error

1.0

END OF TEST

In [78]:
rankings_percentage_float = []
for alist in magic:
    rankings_percentage_float.append(float("{:.2f}".format(100*np.std(alist, dtype=np.float64)/len(alist))))

In [85]:
final_result = str("{:.2f}".format(np.mean(rankings_percentage_float)))

In [88]:
print("\nMean of the ranking deviation: "+final_result+"%")


Mean of the ranking deviation: 5.99%
